In [1]:
import os 
import sys
import requests
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gzip
import shutil
import tempfile
import logging
import pandas as pd
from bs4 import BeautifulSoup
from astropy.io import fits
from PIL import Image
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from datetime import datetime
import time
import random
import uuid
import math
import re
import calendar

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Date validation function
def validate_date_format(date_string):
    clean_date = re.sub(r'[^0-9\-/\.]', '', date_string.strip())
    date_patterns = [
        # DD-MM-YYYY
        r'^(\d{1,2})[\-/\.](\d{1,2})[\-/\.](\d{4})$',
        r'^(\d{4})[\-/\.](\d{1,2})$',
        r'^(\d{4})$'
    ]
    
    for pattern in date_patterns:
        match = re.match(pattern, clean_date)
        if match:
            groups = match.groups()
            
            # Process DD-MM-YYYY 
            if len(groups) == 3 and len(groups[2]) == 4:
                year = int(groups[2])
                # Assume first is month, second is day (MM-DD-YYYY)
                month = int(groups[0])
                day = int(groups[1])
                
                # If month is out of range, try swapping month and day
                if month > 12:
                    month, day = day, month
                
                # Verify again
                if month < 1:
                    month = 1
                elif month > 12:
                    month = 12
                
                max_day = calendar.monthrange(year, month)[1]
                if day < 1:
                    day = 1
                elif day > max_day:
                    day = max_day
                
                return f"{year:04d}-{month:02d}-{day:02d}"
    
    # If no valid format can be recognized
    logger.warning(f"Unable to validate date format: {date_string}")
    return ""


# Attention mechanism module
class Attention(nn.Module):
    def __init__(self, hidden_size):
        super(Attention, self).__init__()
        self.hidden_size = hidden_size
        
        # Attention weight calculation
        self.attn = nn.Linear(hidden_size * 2, hidden_size)
        self.v = nn.Parameter(torch.rand(hidden_size))
        stdv = 1. / math.sqrt(self.v.size(0))
        self.v.data.uniform_(-stdv, stdv)
        
    def forward(self, hidden, encoder_outputs):
        # hidden: [batch_size, hidden_size]
        # encoder_outputs: [batch_size, seq_len, hidden_size]
        
        batch_size = encoder_outputs.size(0)
        seq_len = encoder_outputs.size(1)
        
        # Create reusable hidden
        hidden = hidden.unsqueeze(1).repeat(1, seq_len, 1)
        
        # Concatenate hidden with encoder_outputs, then through attention network
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        
        # Calculate attention weights
        attention = torch.matmul(energy, self.v)
        attention = F.softmax(attention, dim=1)
        
        # Use attention weights to weight encoder_outputs
        context = torch.bmm(attention.unsqueeze(1), encoder_outputs)
        context = context.squeeze(1)
        
        return context, attention

# CRNN model with attention mechanism
class CRNNWithAttention(nn.Module):
    def __init__(self, input_shape, num_classes):
        super(CRNNWithAttention, self).__init__()
        
        # CNN part - same as original CRNN
        self.cnn = nn.Sequential(
            # First layer: input channel 1 -> 32
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Second layer: 32 -> 64
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # Third layer: 64 -> 128
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1)),
            
            # Fourth layer: 128 -> 256
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1)),
        )
        
        self.cnn_dropout = nn.Dropout(0.2)
        
        # Calculate CNN output height
        h, w = input_shape
        h = h // 2 // 2 // 2 // 2  # Four pooling layers
        self.cnn_output_h = h
        
        # RNN part
        self.rnn1 = nn.LSTM(256 * h, 128, bidirectional=True, batch_first=True)
        self.rnn_dropout1 = nn.Dropout(0.2)
        
        # Attention mechanism
        self.attention = Attention(256)  # bidirectional LSTM hidden size = 128*2 = 256
        
        # Second LSTM layer
        self.rnn2 = nn.LSTM(256, 128, bidirectional=True, batch_first=True)
        self.rnn_dropout2 = nn.Dropout(0.2)
        
        # Fully connected layer - increased size to handle attention context
        self.dense = nn.Linear(256 * 2, num_classes)  # 256 from LSTM + 256 from attention
        
    def forward(self, x):
        # CNN feature extraction
        conv = self.cnn(x)
        conv = self.cnn_dropout(conv)
        
        # Prepare RNN input
        batch, channels, height, width = conv.size()
        conv = conv.permute(0, 3, 1, 2)  # (batch, width, channels, height)
        conv = conv.reshape(batch, width, channels * height)
        
        # First LSTM layer
        rnn_out1, (h, c) = self.rnn1(conv)
        rnn_out1 = self.rnn_dropout1(rnn_out1)
        
        # Apply attention mechanism
        h_combined = torch.cat((h[0], h[1]), dim=1)  # Combine bidirectional LSTM hidden states
        context, attention_weights = self.attention(h_combined, rnn_out1)
        
        # Second LSTM layer
        rnn_out2, _ = self.rnn2(rnn_out1)
        rnn_out2 = self.rnn_dropout2(rnn_out2)
        
        # For each time step, concatenate context vector with LSTM output
        context_expanded = context.unsqueeze(1).expand(-1, rnn_out2.size(1), -1)
        combined = torch.cat((rnn_out2, context_expanded), dim=2)
        
        # Final prediction
        output = self.dense(combined)
        output = nn.functional.log_softmax(output, dim=2)
        
        return output

# Label processor class
class LabelProcessor:
    def __init__(self, char_to_num, num_to_char):
        self.char_to_num = char_to_num
        self.num_to_char = num_to_char

# Image processing functions
def fts_to_jpg(image_fts):
    """Convert FITS data to JPG format"""
    # Check data validity
    if image_fts is None or image_fts.size == 0:
        logger.error("FITS data is empty or invalid")
        return np.zeros((600, 800), dtype=np.uint8)  # Return blank image
        
    try:
        min_val = np.min(image_fts)
        max_val = np.max(image_fts)
        
        if np.isnan(min_val) or np.isnan(max_val) or np.isinf(min_val) or np.isinf(max_val):
            logger.warning("FITS data contains NaN or Inf values, replacing")
            # Replace non-finite values with 0
            image_fts = np.nan_to_num(image_fts, nan=0.0, posinf=0.0, neginf=0.0)
            min_val = np.min(image_fts)
            max_val = np.max(image_fts)
            
        if max_val == min_val:
            logger.warning("FITS data has all same values, generating uniform image")
            normalized_array = np.zeros_like(image_fts)
        else:
            normalized_array = (image_fts - min_val) / (max_val - min_val)
            
        jpg_scaled_array = normalized_array * 255
        jpg_converted_array = jpg_scaled_array.astype(np.uint8)
        
        # Check if output is valid
        if jpg_converted_array.size == 0:
            logger.error("Converted JPG data is empty")
            return np.zeros((600, 800), dtype=np.uint8)  # Return blank image
            
        return jpg_converted_array
    except Exception as e:
        logger.error(f"Error converting FITS to JPG: {e}")
        # Return blank image
        return np.zeros((600, 800), dtype=np.uint8)

def resize_with_aspect_ratio(data, target_size=(800, 600)):
    """Resize image while maintaining aspect ratio"""
    try:
        # Check if input data is valid
        if data is None or data.size == 0 or data.ndim < 2:
            logger.error(f"Invalid image data: shape={data.shape if data is not None else None}")
            return np.zeros(target_size[::-1], dtype=np.uint8)  # Return blank image
            
        # Ensure data is 2D
        if data.ndim > 2:
            data = cv2.cvtColor(data, cv2.COLOR_BGR2GRAY)
            
        # Use OpenCV for resizing - more stable method
        resized_data = cv2.resize(data, (target_size[0], target_size[1]), interpolation=cv2.INTER_LANCZOS4)
        
        # Validate output
        if resized_data.shape != target_size[::-1]:  # Note OpenCV uses (width, height) while numpy uses (height, width)
            logger.warning(f"Resized image shape mismatch: expected {target_size[::-1]}, got {resized_data.shape}")
            # Force resize to correct size
            resized_data = cv2.resize(data, (target_size[0], target_size[1]), interpolation=cv2.INTER_NEAREST)
            
        return resized_data
    except Exception as e:
        logger.error(f"Error resizing image: {e}")
        # Return blank image
        return np.zeros(target_size[::-1], dtype=np.uint8)

# Same decoding function as during training
def greedy_decode_predictions(predictions, label_processor):
    """Use greedy search for CTC decoding - same as during training"""
    pred_indices = predictions.argmax(dim=2).cpu().numpy()
    
    decoded_labels = []
    for pred in pred_indices:
        # Apply CTC decoding rules:
        # 1. Remove repeated characters
        # 2. Remove blank characters (index 0)
        filtered = []
        last_char = -1
        for p in pred:
            if p != 0 and p != last_char:  # Not blank and not repeated
                filtered.append(p)
                last_char = p
        
        # Convert to text
        text = ''.join([label_processor.num_to_char.get(i, '') for i in filtered])
        decoded_labels.append(text)
    
    return decoded_labels

# Preprocessing function 
def preprocess_for_ocr(cropped_image, target_shape):
    """Image preprocessing function identical to training"""
    try:
        h, w = target_shape
        
        # Check input image
        if cropped_image is None or cropped_image.size == 0:
            logger.error("Cropped image is empty")
            return np.zeros(target_shape, dtype=np.float32)
        
        # Ensure image is 2D
        if cropped_image.ndim > 2:
            cropped_image = cv2.cvtColor(cropped_image, cv2.COLOR_BGR2GRAY)
        
        # 1. Rotate image - same as during training
        rotated_image = cv2.rotate(cropped_image, cv2.ROTATE_90_COUNTERCLOCKWISE)
        
        # 2. Padding processing - exactly the same as during training
        original_height, original_width = rotated_image.shape
        pad_height = h - original_height
        pad_width = w - original_width
        
        # Ensure padding is not negative
        pad_height = max(0, pad_height)
        pad_width = max(0, pad_width)
        
        top_pad = pad_height // 2
        bottom_pad = pad_height - top_pad
        left_pad = pad_width // 2
        right_pad = pad_width - left_pad
        
        # Apply padding
        if pad_height > 0 or pad_width > 0:
            padded_image = np.pad(
                rotated_image,
                ((top_pad, bottom_pad), 
                (left_pad, right_pad)),
                mode='constant',
                constant_values=0
            )
        else:
            # If we need to shrink, not pad
            padded_image = cv2.resize(rotated_image, (w, h), interpolation=cv2.INTER_LANCZOS4)
        
        # 3. Normalize to [0, 1] - exactly the same as during training
        processed_image = padded_image.astype(np.float32) / 255.0
        
        # Ensure output shape is correct
        if processed_image.shape != target_shape:
            logger.warning(f"Preprocessed image shape doesn't match expected: {processed_image.shape} vs {target_shape}")
            processed_image = cv2.resize(processed_image, (target_shape[1], target_shape[0]))
            processed_image = processed_image.astype(np.float32) # Ensure correct type
        
        return processed_image
    except Exception as e:
        logger.error(f"Error during OCR preprocessing: {e}")
        return np.zeros(target_shape, dtype=np.float32)

# Download and process image
def download_and_process_image(url, save_dir=None, max_retries=3):
    for attempt in range(max_retries):
        try:
            filename = os.path.basename(url)
            
            logger.info(f"Downloading {filename} (attempt {attempt+1}/{max_retries})")
            response = requests.get(url, stream=True, timeout=30)
            response.raise_for_status()
            
            # Use unique filename to avoid conflicts
            unique_id = str(uuid.uuid4())[:8]
            temp_gz_path = os.path.join(tempfile.gettempdir(), f"solar_{unique_id}_{filename}")
            extracted_fits_file = temp_gz_path.replace('.gz', '')
            
            # Download to temporary file
            with open(temp_gz_path, 'wb') as f_out:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f_out.write(chunk)
            
            # Decompress
            try:
                with gzip.open(temp_gz_path, 'rb') as f_in:
                    with open(extracted_fits_file, 'wb') as f_out:
                        shutil.copyfileobj(f_in, f_out)
            except gzip.BadGzipFile:
                logger.error(f"Invalid gzip file: {temp_gz_path}")
                # Try using file directly
                shutil.copyfile(temp_gz_path, extracted_fits_file)
            
            # Open FITS file
            try:
                hdul = fits.open(extracted_fits_file)
                data = hdul[0].data
                hdul.close()
            except Exception as e:
                logger.error(f"Error reading FITS file: {e}")
                # Try reading directly as image
                try:
                    data = np.array(Image.open(extracted_fits_file).convert('L'))
                except:
                    # If all fails, create a blank image
                    logger.warning("Creating blank image as fallback")
                    data = np.zeros((600, 800), dtype=np.uint8)
            
            # Process image
            try:
                # Check if data is None or empty
                if data is None or data.size == 0:
                    logger.warning("Image data is empty, creating blank image")
                    data = np.zeros((600, 800), dtype=np.uint8)
                else:
                    # Ensure data is 2D
                    if data.ndim < 2:
                        logger.warning(f"Image dimensions less than 2: {data.ndim}")
                        data = np.zeros((600, 800), dtype=np.uint8)
                    elif data.ndim > 2:
                        logger.info(f"Converting {data.ndim}D image to 2D")
                        if data.ndim == 3 and data.shape[2] == 3:  # RGB image
                            data = cv2.cvtColor(data, cv2.COLOR_RGB2GRAY)
                        else:
                            data = data[..., 0]  # Take first channel
                
                # Flip and convert
                data = np.flipud(data)
                jpg_data = fts_to_jpg(data)
                resized_data = resize_with_aspect_ratio(jpg_data, target_size=(800, 600))
                
                # Check processed image
                if resized_data is None or resized_data.size == 0:
                    logger.warning("Processed image is empty, creating blank image")
                    resized_data = np.zeros((600, 800), dtype=np.uint8)
                    
                # Ensure image dimensions are correct
                if resized_data.shape != (600, 800):
                    logger.warning(f"Image shape incorrect: {resized_data.shape}, resizing to (600, 800)")
                    resized_data = cv2.resize(resized_data, (800, 600))
            except Exception as e:
                logger.error(f"Error processing image data: {e}")
                resized_data = np.zeros((600, 800), dtype=np.uint8)
            
            # Save processed image
            if save_dir:
                os.makedirs(save_dir, exist_ok=True)
                save_path = os.path.join(save_dir, filename.replace('.fts.gz', '.jpg'))
                try:
                    Image.fromarray(resized_data).save(save_path)
                    logger.info(f"Image saved: {save_path}")
                except Exception as e:
                    logger.error(f"Error saving image: {e}")
            
            # Clean up temp files
            try:
                if os.path.exists(temp_gz_path):
                    os.remove(temp_gz_path)
                if os.path.exists(extracted_fits_file):
                    os.remove(extracted_fits_file)
            except Exception as e:
                logger.warning(f"Error cleaning up temp files: {e}")
            
            return resized_data, filename
            
        except Exception as e:
            logger.warning(f"Error processing image {url} (attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(random.uniform(1, 3))
    
    logger.error(f"Failed to process image after maximum retries: {url}")
    # Return blank image and filename
    return np.zeros((600, 800), dtype=np.uint8), os.path.basename(url)

# Model loading
def load_models(yolo_model_path, ocr_model_paths):
    try:
        # Try to import YOLO
        try:
            from ultralytics import YOLO
            yolo_model = YOLO(yolo_model_path)
            logger.info(f"YOLO model loaded: {yolo_model_path}")
        except ImportError:
            logger.error("ultralytics library not installed, please install with pip install ultralytics")
            return None, None
        except Exception as e:
            logger.error(f"Error loading YOLO model: {e}")
            return None, None
        
        ocr_models = {}
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        for field, model_path in ocr_model_paths.items():
            if not os.path.exists(model_path):
                logger.error(f"Model does not exist: {model_path}")
                continue
                
            logger.info(f"Loading OCR model {field}: {model_path}")
            
            try:
                # Load checkpoint
                checkpoint = torch.load(model_path, map_location=device)
                
                # Extract and validate necessary model parameters
                char_to_num = checkpoint.get('char_to_num')
                num_to_char = checkpoint.get('num_to_char')
                input_shape = checkpoint.get('input_shape')
                num_classes = checkpoint.get('num_classes')
                # Check for model type - accept either crnn or crnn_attention
                model_type = checkpoint.get('model_type', 'crnn_attention')
                
                if not all([char_to_num, num_to_char, input_shape, num_classes]):
                    logger.error(f"Model {field} missing necessary parameters")
                    continue
                
                # Print model info for verification
                logger.info(f"Model {field} - charset size: {len(char_to_num)}, input shape: {input_shape}, classes: {num_classes}, type: {model_type}")
                
                # Create label processor same as during training
                label_processor = LabelProcessor(char_to_num, num_to_char)
                
                # Create model structure same as during training - now using CRNNWithAttention
                model = CRNNWithAttention(input_shape, num_classes)
                
                # If the checkpoint was saved with the original CRNN structure, we need to handle parameter mismatch
                if model_type == 'crnn':
                    logger.warning(f"Model {field} was trained with regular CRNN, adapting parameters for attention version")
                    # Create a temporary CRNN model to get its state dict
                    # We would need to define the original CRNN class here, but for simplicity, 
                    # we'll assume the checkpoint's state_dict keys can be loaded partially
                    
                    # Try to load with strict=False to allow parameter mismatch
                    model.load_state_dict(checkpoint['model_state_dict'], strict=False)
                    
                    # Log warning about parameter mismatch
                    logger.warning("Some parameters may not be loaded correctly due to model structure differences")
                else:
                    # Standard loading for models already trained with attention
                    model.load_state_dict(checkpoint['model_state_dict'])
                
                model = model.to(device)
                model.eval()  # Set to evaluation mode
                
                # Save model and related components
                ocr_models[field] = {
                    'model': model,
                    'label_processor': label_processor,
                    'input_shape': input_shape,
                    'char_to_num': char_to_num,
                    'num_to_char': num_to_char
                }
                
                logger.info(f"Model {field} loaded successfully")
            except Exception as e:
                logger.error(f"Error loading OCR model {field}: {e}")
        
        if not ocr_models:
            logger.error("No OCR models loaded successfully")
            return None, None
            
        return yolo_model, ocr_models
    except Exception as e:
        logger.error(f"Error loading models: {e}")
        return None, None

# Function to create debug visualization for model prediction
def debug_model_prediction(model, image, label_processor, input_shape, device, true_text=None):
    """Debug OCR model prediction results"""
    try:
        # Create figure
        plt.figure(figsize=(10, 10))
        
        # 1. Visualize original image
        plt.subplot(2, 2, 1)
        plt.imshow(image, cmap='gray')
        plt.title("Original Image")
        
        # 2. Preprocess image
        processed_image = preprocess_for_ocr(image, input_shape)
        plt.subplot(2, 2, 2)
        plt.imshow(processed_image, cmap='gray')
        plt.title(f"Preprocessed Image (shape: {processed_image.shape})")
        
        # 3. Inference
        image_tensor = torch.FloatTensor(processed_image).unsqueeze(0).unsqueeze(0).to(device)
        
        with torch.no_grad():
            predictions = model(image_tensor)
            decoded_text = greedy_decode_predictions(predictions, label_processor)[0]
        
        # 4. Visualize probability distribution
        pred_probs = torch.exp(predictions).cpu().detach().numpy()[0]
        plt.subplot(2, 2, 3)
        plt.imshow(pred_probs.T, aspect='auto', cmap='viridis')
        plt.colorbar()
        plt.title(f"Prediction Probabilities (shape: {pred_probs.shape})")
        
        # 5. Display results
        result_title = f"Prediction: '{decoded_text}'"
        if true_text:
            result_title += f"\nGround Truth: '{true_text}'"
            if decoded_text == true_text:
                result_title += " ✓"
            else:
                result_title += " ✗"
        
        plt.subplot(2, 2, 4)
        plt.axis('off')
        plt.text(0.1, 0.5, result_title, fontsize=12)
        plt.title("Prediction Results")
        
        plt.tight_layout()
        plt.savefig("debug_prediction.png")
        plt.close()
        
        return decoded_text, "debug_prediction.png"
    except Exception as e:
        logger.error(f"Error debugging prediction: {e}")
        return "", None

# Text recognition function
def process_and_recognize(image, yolo_model, ocr_models, conf_threshold=0.25, debug_mode=False):
    try:
        # Check if image is valid
        if image is None or image.size == 0:
            logger.error("Input image is empty or invalid")
            return {}, {}, np.zeros((600, 800), dtype=np.uint8)
            
        # Ensure image shape is correct
        if image.shape != (600, 800) and image.ndim == 2:
            logger.warning(f"Image shape incorrect: {image.shape}, resizing to (600, 800)")
            image = cv2.resize(image, (800, 600))
            
        # If image is grayscale, convert to RGB for YOLO
        if image.ndim == 2:
            image_rgb = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        elif image.ndim == 3 and image.shape[2] == 1:
            image_rgb = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        else:
            image_rgb = image
            
        logger.info(f"Running YOLO detection, image shape: {image_rgb.shape}")
        
        # Run YOLO detection
        results = yolo_model.predict(image_rgb, conf=conf_threshold)
        
        if len(results) == 0 or not hasattr(results[0], 'boxes'):
            logger.warning("YOLO returned no detection results")
            return {}, {}, image
        
        boxes = results[0].boxes
        if len(boxes) == 0:
            logger.warning("No bounding boxes detected")
            return {}, {}, image
            
        class_ids = boxes.cls.cpu().numpy()
        confidence = boxes.conf.cpu().numpy()
        coordinates = boxes.xyxy.cpu().numpy()
        
        class_names = results[0].names
        recognized_data = {}
        detected_boxes = {}
        debug_images = {}
        
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        for i, (class_id, conf, coord) in enumerate(zip(class_ids, confidence, coordinates)):
            class_name = class_names[int(class_id)]
            
            if class_name == 'background':
                continue
            
            x1, y1, x2, y2 = map(int, coord)
            
            # Validate coordinates
            if x1 >= x2 or y1 >= y2 or x1 < 0 or y1 < 0 or x2 > image.shape[1] or y2 > image.shape[0]:
                logger.warning(f"Invalid bounding box coordinates: {(x1, y1, x2, y2)}")
                continue
                
            width, height = x2 - x1, y2 - y1
            margin_x, margin_y = int(width * 0.1), int(height * 0.1)
            
            img_h, img_w = image.shape[:2] if image.ndim > 1 else (image.shape[0], image.shape[1])
            x1 = max(0, x1 - margin_x)
            y1 = max(0, y1 - margin_y)
            x2 = min(img_w, x2 + margin_x)
            y2 = min(img_h, y2 + margin_y)
            
            # Ensure image is 2D for cropping
            if image.ndim == 3:
                crop_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            else:
                crop_image = image
                
            # Crop region
            try:
                cropped_region = crop_image[y1:y2, x1:x2]
                
                # Check if cropping was successful
                if cropped_region.size == 0:
                    logger.warning(f"Cropped region is empty: {(x1, y1, x2, y2)}")
                    continue
            except Exception as e:
                logger.error(f"Error cropping image: {e}, coordinates: {(x1, y1, x2, y2)}, image shape: {crop_image.shape}")
                continue
            
            detected_boxes[class_name] = {
                'coordinates': (x1, y1, x2, y2),
                'confidence': conf
            }
            
            if class_name in ocr_models:
                ocr_info = ocr_models[class_name]
                ocr_model = ocr_info['model']
                label_processor = ocr_info['label_processor']
                input_shape = ocr_info['input_shape']
                
                if debug_mode:
                    # In debug mode, save more intermediate results
                    debug_text, debug_image_path = debug_model_prediction(
                        ocr_model, cropped_region, label_processor, input_shape, device
                    )
                    recognized_data[class_name] = debug_text
                    if debug_image_path:
                        debug_images[class_name] = debug_image_path
                else:
                    # Normal mode - preprocess cropped region
                    processed_image = preprocess_for_ocr(cropped_region, input_shape)
                    
                    # Convert to tensor
                    image_tensor = torch.FloatTensor(processed_image).unsqueeze(0).unsqueeze(0)
                    image_tensor = image_tensor.to(device)
                    
                    # Validate tensor shape
                    expected_shape = (1, 1, input_shape[0], input_shape[1])
                    if image_tensor.shape != expected_shape:
                        logger.warning(f"Tensor shape mismatch: {image_tensor.shape} vs {expected_shape}")
                        # Adjust shape
                        image_tensor = torch.zeros(expected_shape, device=device)
                    
                    # Run OCR prediction
                    with torch.no_grad():
                        try:
                            predictions = ocr_model(image_tensor)
                            decoded_text = greedy_decode_predictions(predictions, label_processor)[0]
                            if class_name == 'date':
                                validated_text = validate_date_format(decoded_text)
                                if validated_text:
                                    if validated_text != decoded_text:
                                        logger.info(f"Date corrected: '{decoded_text}' -> '{validated_text}'")
                                    decoded_text = validated_text
                            recognized_data[class_name] = decoded_text
                            logger.info(f"Recognized {class_name}: {decoded_text} (confidence: {conf:.2f})")
                        except Exception as e:
                            logger.error(f"Error during OCR prediction: {e}")
                            recognized_data[class_name] = ""
            else:
                logger.warning(f"No OCR model available for {class_name}")
                recognized_data[class_name] = "Unrecognized"
        
        if debug_mode:
            return recognized_data, detected_boxes, image, debug_images
        else:
            return recognized_data, detected_boxes, image
    except Exception as e:
        logger.error(f"Error during recognition process: {e}")
        return {}, {}, image

# Safely save results
def save_results_safely(image, recognized_data, detected_boxes, filename, output_dir, debug_images=None):
    """Safely save results"""
    try:
        # Check if image is valid
        if image is None or image.size == 0:
            logger.error("Visualization image is empty or invalid")
            image = np.zeros((600, 800), dtype=np.uint8)
        
        # Ensure image is RGB
        if image.ndim == 2:
            result_image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
        elif image.ndim == 3 and image.shape[2] == 1:
            result_image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
        elif image.ndim == 3 and image.shape[2] == 3:
            result_image = image.copy()
        else:
            logger.warning(f"Unexpected image shape: {image.shape}, creating blank image")
            result_image = np.zeros((600, 800, 3), dtype=np.uint8)
        
        # Add bounding boxes and labels
        colors = {
            'rell_number': (0, 255, 0),    # Green
            'date': (255, 0, 0),           # Red
            'hour': (0, 0, 255),           # Blue
            'minute': (255, 255, 0),       # Yellow
            'seconds': (0, 255, 255),      # Cyan
            'filter': (255, 0, 255)        # Magenta
        }
        
        for class_name, box_info in detected_boxes.items():
            x1, y1, x2, y2 = box_info['coordinates']
            conf = box_info['confidence']
            
            color = colors.get(class_name, (255, 255, 255))
            cv2.rectangle(result_image, (x1, y1), (x2, y2), color, 2)
            
            if class_name in recognized_data:
                text = f"{class_name}: {recognized_data[class_name]}"
            else:
                text = f"{class_name}"
            
            cv2.putText(result_image, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        
        # Add title text
        title_text = f"File: {filename}"
        cv2.putText(result_image, title_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
        # Add recognition results summary
        y_pos = 60
        if recognized_data:
            for class_name, text in recognized_data.items():
                summary_text = f"{class_name}: {text}"
                cv2.putText(result_image, summary_text, (10, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
                y_pos += 25
        else:
            cv2.putText(result_image, "No text detected", (10, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        # Create results directory
        os.makedirs(output_dir, exist_ok=True)
        
        # Save result image
        result_path = os.path.join(output_dir, f"{filename.replace('.fts.gz', '')}_result.jpg")
        cv2.imwrite(result_path, cv2.cvtColor(result_image, cv2.COLOR_RGB2BGR))
        logger.info(f"Result saved: {result_path}")
        
        # If debug images exist, copy to output directory
        debug_paths = {}
        if debug_images:
            for class_name, debug_path in debug_images.items():
                if os.path.exists(debug_path):
                    # Copy debug image to output directory
                    debug_output = os.path.join(output_dir, f"{filename.replace('.fts.gz', '')}_{class_name}_debug.jpg")
                    shutil.copy(debug_path, debug_output)
                    debug_paths[class_name] = debug_output
        
        return result_path, debug_paths
    except Exception as e:
        logger.error(f"Error saving results: {e}")
        return None, {}

# Generate CSV report
def generate_csv_report(results, output_path):
    data = []
    
    for filename, file_data in results.items():
        recognized = file_data.get('recognized_data', {})
        row = {'filename': filename}
        
        for field, text in recognized.items():
            if field == 'date':
                validated_text = validate_date_format(text)
                if validated_text:
                    text = validated_text
            row[field] = text
        
        data.append(row)
    
    if data:
        df = pd.DataFrame(data)
        try:
            df.to_csv(output_path, index=False, encoding='utf-8')
            logger.info(f"CSV report saved: {output_path}")
        except Exception as e:
            logger.error(f"Error saving CSV report: {e}")

# Get image URLs from directory
def get_image_urls(url, limit=5):
    if not url.endswith('/'):
        url += '/'
    
    try:
        max_retries = 3
        response = None
        
        for attempt in range(max_retries):
            try:
                response = requests.get(url, timeout=30)
                if response.status_code == 200:
                    break
                logger.warning(f"Failed to get URL content (attempt {attempt+1}/{max_retries}): HTTP {response.status_code}")
                if attempt < max_retries - 1:
                    time.sleep(random.uniform(1, 3))
            except requests.exceptions.RequestException as e:
                logger.warning(f"Request failed (attempt {attempt+1}/{max_retries}): {e}")
                if attempt < max_retries - 1:
                    time.sleep(random.uniform(1, 3))
        
        if response is None or response.status_code != 200:
            logger.error(f"Cannot access URL: {url}")
            return []
        
        soup = BeautifulSoup(response.text, 'html.parser')
        image_urls = []
        
        for link in soup.find_all('a'):
            href = link.get('href')
            if href and href.endswith('.fts.gz'):
                full_url = url + href
                image_urls.append(full_url)
        
        # If no .fts.gz files found, try other possible extensions
        if not image_urls:
            logger.warning(f"No .fts.gz files found in {url}, trying other extensions")
            for link in soup.find_all('a'):
                href = link.get('href')
                if href and (href.endswith('.fts') or href.endswith('.fits') or 
                            href.endswith('.FITS') or href.endswith('.FTS')):
                    full_url = url + href
                    image_urls.append(full_url)
        
        logger.info(f"Found {len(image_urls)} images, will process first {min(limit, len(image_urls))}")
        return image_urls[:limit]
    
    except Exception as e:
        logger.error(f"Error getting image URLs: {e}")
        return []

# Process single image
def process_single_image(url, yolo_model, ocr_models, output_dir="nisp_results", debug_mode=False):
    try:
        os.makedirs(output_dir, exist_ok=True)
        results_dir = os.path.join(output_dir, "results")
        os.makedirs(results_dir, exist_ok=True)
        
        logger.info(f"Processing single image: {url}")
        processed_image, filename = download_and_process_image(url, save_dir=output_dir)
        
        if processed_image is None:
            logger.error(f"Failed to process image: {url}")
            return {}
        
        # Check image
        logger.info(f"Processed image shape: {processed_image.shape}, type: {processed_image.dtype}")
        
        # Perform recognition
        if debug_mode:
            recognized_data, detected_boxes, processed_image, debug_images = process_and_recognize(
                processed_image, yolo_model, ocr_models, debug_mode=True)
        else:
            recognized_data, detected_boxes, processed_image = process_and_recognize(
                processed_image, yolo_model, ocr_models)
            debug_images = None
        
        if not detected_boxes:
            logger.warning(f"No text regions detected: {url}")
        
        # Safely save results
        if debug_mode:
            result_path, debug_paths = save_results_safely(processed_image, recognized_data, detected_boxes, 
                                            filename, results_dir, debug_images)
        else:
            result_path, _ = save_results_safely(processed_image, recognized_data, detected_boxes, 
                                            filename, results_dir)
        
        result = {
            filename: {
                'recognized_data': recognized_data,
                'result_image_path': result_path
            }
        }
        
        if debug_mode and 'debug_paths' in locals():
            result[filename]['debug_image_paths'] = debug_paths
        
        generate_csv_report(result, os.path.join(output_dir, "result.csv"))
        
        return result
    except Exception as e:
        logger.error(f"Error processing single image: {e}")
        return {}

# Process multiple images from directory
def process_directory(url, yolo_model, ocr_models, output_dir="nisp_results", limit=5, debug_mode=False):
    try:
        os.makedirs(output_dir, exist_ok=True)
        results_dir = os.path.join(output_dir, "results")
        os.makedirs(results_dir, exist_ok=True)
        
        # Get image URLs
        logger.info(f"Getting image URLs from directory: {url}")
        image_urls = get_image_urls(url, limit)
        
        if not image_urls:
            logger.warning(f"No images found in {url}")
            return {}
        
        all_results = {}
        
        for i, img_url in enumerate(image_urls):
            try:
                logger.info(f"Processing image {i+1}/{len(image_urls)}: {img_url}")
                processed_image, filename = download_and_process_image(img_url, save_dir=output_dir)
                
                if processed_image is None:
                    continue
                
                # Perform recognition
                if debug_mode:
                    recognized_data, detected_boxes, processed_image, debug_images = process_and_recognize(
                        processed_image, yolo_model, ocr_models, debug_mode=True)
                else:
                    recognized_data, detected_boxes, processed_image = process_and_recognize(
                        processed_image, yolo_model, ocr_models)
                    debug_images = None
                
                # Safely save results
                if debug_mode:
                    result_path, debug_paths = save_results_safely(processed_image, recognized_data, detected_boxes,
                                                                filename, results_dir, debug_images)
                else:
                    result_path, _ = save_results_safely(processed_image, recognized_data, detected_boxes,
                                                     filename, results_dir)
                
                result_info = {
                    'recognized_data': recognized_data,
                    'result_image_path': result_path
                }
                
                if debug_mode and 'debug_paths' in locals():
                    result_info['debug_image_paths'] = debug_paths
                    
                all_results[filename] = result_info
            except Exception as e:
                logger.error(f"Error processing image {img_url}: {e}")
        
        if all_results:
            generate_csv_report(all_results, os.path.join(output_dir, "results.csv"))
        
        return all_results
    except Exception as e:
        logger.error(f"Error processing directory: {e}")
        return {}


def recognize_solar_image(url, output_dir="nisp_results", limit=5, 
                         yolo_model_path=None, ocr_models_dir=None, show_images=False, debug_mode=False):
    """
    Recognize text in solar flare images
    
    Args:
        url: URL to process (file or directory)
        output_dir: Output directory
        limit: Maximum number of files to process if url is a directory
        yolo_model_path: YOLO model path (if None, will look in common paths)
        ocr_models_dir: OCR models directory (if None, will look in common paths)
        show_images: Whether to display processed images (set to False in Jupyter to avoid crashes)
        debug_mode: Whether to enable debug mode
    
    Returns:
        Processing results
    """
    # Find YOLO model path
    if yolo_model_path is None:
        # List of common model paths
        possible_yolo_paths = [
            "yolo_train_results/train6/weights/best.pt",
            "yolo_train_results/train/weights/best.pt",
            "image_split/yolo_train_results/train6/weights/best.pt",
            "E:/pythonfile/main-task/image_split/yolo_train_results/train6/weights/best.pt",
            os.path.join(os.getcwd(), "yolo_train_results/train6/weights/best.pt")
        ]
        
        for path in possible_yolo_paths:
            if os.path.exists(path):
                yolo_model_path = path
                logger.info(f"Found YOLO model: {path}")
                break
        
        if yolo_model_path is None:
            logger.error("No YOLO model found, please specify path manually")
            return None
    
    # Find OCR models directory
    if ocr_models_dir is None:
        # List of common OCR model directories
        possible_ocr_dirs = [
            ".",
            "read text",
            "E:/pythonfile/main-task/read text",
            os.getcwd()
        ]
        
        for dir_path in possible_ocr_dirs:
            rell_path = os.path.join(dir_path, "rell_number_crnn_attention_models/rell_number_crnn_attention_best_model.pt")
            if os.path.exists(rell_path):
                ocr_models_dir = dir_path
                logger.info(f"Found OCR models directory: {dir_path}")
                break
            '''
            # Also check for standard model path
            rell_path_std = os.path.join(dir_path, "rell_number_models/rell_number_best_model.pt")
            if os.path.exists(rell_path_std):
                ocr_models_dir = dir_path
                logger.info(f"Found OCR models directory (standard): {dir_path}")
                break
            '''
        if ocr_models_dir is None:
            logger.error("No OCR models directory found, please specify path manually")
            return None
    
    # Set OCR model paths - try attention models first, fall back to standard models
    field_names = ['rell_number', 'date', 'hour', 'minute', 'seconds']
    ocr_model_paths = {}
    
    for field in field_names:
        # First try the attention-specific model path
        attn_path = os.path.join(ocr_models_dir, f"{field}_crnn_attention_models/{field}_crnn_attention_best_model.pt")
        #std_path = os.path.join(ocr_models_dir, f"{field}_models/{field}_best_model.pt")
        
        ocr_model_paths[field] = attn_path
        logger.info(f"Using attention model for {field}: {attn_path}")
        '''
        if os.path.exists(attn_path):
            ocr_model_paths[field] = attn_path
            logger.info(f"Using attention model for {field}: {attn_path}")
        elif os.path.exists(std_path):
            ocr_model_paths[field] = std_path
            logger.info(f"Using standard model for {field}: {std_path}")
        else:
            logger.warning(f"No model found for {field}")
        '''
    # Load models
    logger.info("Loading models...")
    yolo_model, ocr_models = load_models(yolo_model_path, ocr_model_paths)
    
    if yolo_model is None or ocr_models is None:
        logger.error("Failed to load models")
        return None
    
    # Check if URL is a file or directory
    if url.endswith(('.fts.gz', '.fts', '.fits', '.FITS', '.FTS')):
        # Process single file
        results = process_single_image(url, yolo_model, ocr_models, output_dir, debug_mode=debug_mode)
    else:
        # Process directory
        results = process_directory(url, yolo_model, ocr_models, output_dir, limit, debug_mode=debug_mode)
    
    # If showing result images is requested
    if show_images and results:
        try:
            # Display at most the first 3 results using non-interactive backend
            count = 0
            for filename, file_data in results.items():
                if count >= 3:  # Limit display count
                    break
                
                result_path = file_data.get('result_image_path')
                if result_path and os.path.exists(result_path):
                    # Show result image
                    img = plt.imread(result_path)
                    plt.figure(figsize=(10, 6))
                    plt.imshow(img)
                    plt.axis('off')
                    plt.title(f"Processing Result: {filename}")
                    plt.show()
                    count += 1
        except Exception as e:
            logger.error(f"Error displaying result images: {e}")
    
    return results

# Add testing and debugging function
def test_model_accuracy(model_path, test_images, device=None):
    """Test model accuracy on given image set
    
    Args:
        model_path: Model path
        test_images: List of (image, true_text) tuples
        device: Compute device, default None (auto select)
    """
    # Set device
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    try:
        # Load checkpoint
        checkpoint = torch.load(model_path, map_location=device)
        
        # Extract necessary parameters
        char_to_num = checkpoint.get('char_to_num')
        num_to_char = checkpoint.get('num_to_char')
        input_shape = checkpoint.get('input_shape')
        num_classes = checkpoint.get('num_classes')
        model_type = checkpoint.get('model_type', 'crnn')
        
        # Validate parameters
        if not all([char_to_num, num_to_char, input_shape, num_classes]):
            logger.error(f"Model missing necessary parameters")
            return
            
        logger.info(f"Loading model: {model_path}")
        logger.info(f"Charset size: {len(char_to_num)}")
        logger.info(f"Input shape: {input_shape}")
        logger.info(f"Class count: {num_classes}")
        logger.info(f"Model type: {model_type}")
        
        # Create appropriate model based on model_type
        if model_type == 'crnn_attention':
            logger.info("Creating CRNN with Attention model")
            model = CRNNWithAttention(input_shape, num_classes)
        else:
            logger.warning("Original model was standard CRNN, but we're using CRNN with Attention")
            model = CRNNWithAttention(input_shape, num_classes)
        
        # Load model weights - with error handling for structure differences
        try:
            model.load_state_dict(checkpoint['model_state_dict'])
        except Exception as e:
            logger.warning(f"Error loading model state: {e}")
            logger.warning("Attempting to load with strict=False")
            model.load_state_dict(checkpoint['model_state_dict'], strict=False)
            
        model = model.to(device)
        model.eval()
        
        # Create label processor
        label_processor = LabelProcessor(char_to_num, num_to_char)
        
        # Test accuracy
        correct = 0
        total = 0
        results = []
        
        for i, (image, true_text) in enumerate(test_images):
            # Preprocess image
            processed_image = preprocess_for_ocr(image, input_shape)
            
            # Convert to tensor
            image_tensor = torch.FloatTensor(processed_image).unsqueeze(0).unsqueeze(0).to(device)
            
            # Predict
            with torch.no_grad():
                predictions = model(image_tensor)
                decoded_text = greedy_decode_predictions(predictions, label_processor)[0]
            
            # Check correctness
            is_correct = (decoded_text == true_text)
            if is_correct:
                correct += 1
            total += 1
            
            # Save results
            results.append({
                'image_idx': i,
                'true_text': true_text,
                'predicted_text': decoded_text,
                'is_correct': is_correct
            })
            
            # Display results
            logger.info(f"Sample {i+1}: True='{true_text}', Predicted='{decoded_text}', {'✓' if is_correct else '✗'}")
            
            # Visualize
            plt.figure(figsize=(10, 5))
            plt.subplot(1, 2, 1)
            plt.imshow(image, cmap='gray')
            plt.title(f"Original Image")
            
            plt.subplot(1, 2, 2)
            plt.imshow(processed_image, cmap='gray')
            color = 'green' if is_correct else 'red'
            plt.title(f"Processed: Pred='{decoded_text}'\nTrue='{true_text}'", color=color)
            
            plt.savefig(f"test_sample_{i}.png")
            plt.close()
        
        # Print overall results
        accuracy = correct / total if total > 0 else 0
        logger.info(f"\nOverall Accuracy: {accuracy:.2%} ({correct}/{total})")
        
        return accuracy, results
    
    except Exception as e:
        logger.error(f"Error testing model: {e}")
        import traceback
        traceback.print_exc()
        return 0, []

if __name__ == "__main__":
    # This code won't execute when run directly in a Jupyter cell
    print("Use the recognize_solar_image function to perform recognition")

Use the recognize_solar_image function to perform recognition


In [2]:

recognize_solar_image(
    "https://nispdata.nso.edu/ftp/flare_patrol_h_alpha_sp/fts/223/",
    output_dir="my_results",
    limit=10  
)

2025-05-12 07:30:52,743 - INFO - Found YOLO model: E:/pythonfile/main-task/image_split/yolo_train_results/train6/weights/best.pt
2025-05-12 07:30:52,746 - INFO - Found OCR models directory: E:/pythonfile/main-task/read text
2025-05-12 07:30:52,747 - INFO - Using attention model for rell_number: E:/pythonfile/main-task/read text\rell_number_crnn_attention_models/rell_number_crnn_attention_best_model.pt
2025-05-12 07:30:52,749 - INFO - Using attention model for date: E:/pythonfile/main-task/read text\date_crnn_attention_models/date_crnn_attention_best_model.pt
2025-05-12 07:30:52,751 - INFO - Using attention model for hour: E:/pythonfile/main-task/read text\hour_crnn_attention_models/hour_crnn_attention_best_model.pt
2025-05-12 07:30:52,753 - INFO - Using attention model for minute: E:/pythonfile/main-task/read text\minute_crnn_attention_models/minute_crnn_attention_best_model.pt
2025-05-12 07:30:52,754 - INFO - Using attention model for seconds: E:/pythonfile/main-task/read text\seconds


0: 480x640 (no detections), 50.8ms
Speed: 22.9ms preprocess, 50.8ms inference, 38.4ms postprocess per image at shape (1, 3, 480, 640)


2025-05-12 07:34:02,500 - WARNING - No bounding boxes detected
2025-05-12 07:34:02,563 - INFO - Result saved: my_results\results\nso_0223_00000_result.jpg
2025-05-12 07:34:02,564 - INFO - Processing image 2/10: https://nispdata.nso.edu/ftp/flare_patrol_h_alpha_sp/fts/223/nso_0223_00001.fts.gz
2025-05-12 07:34:02,565 - INFO - Downloading nso_0223_00001.fts.gz (attempt 1/3)
2025-05-12 07:34:07,072 - INFO - Image saved: my_results\nso_0223_00001.jpg
2025-05-12 07:34:07,077 - WARNING - Error cleaning up temp files: [WinError 32] 另一个程序正在使用此文件，进程无法访问。: 'C:\\Users\\kuinan\\AppData\\Local\\Temp\\solar_4312de9f_nso_0223_00001.fts'
2025-05-12 07:34:07,083 - INFO - Running YOLO detection, image shape: (600, 800, 3)



0: 480x640 1 rell_number, 1 date, 1 hour, 1 minute, 22.0ms
Speed: 5.0ms preprocess, 22.0ms inference, 85.6ms postprocess per image at shape (1, 3, 480, 640)


2025-05-12 07:34:09,830 - INFO - Recognized minute: 58 (confidence: 0.88)
2025-05-12 07:34:09,832 - WARNING - Preprocessed image shape doesn't match expected: (64, 146) vs (64, 136)
2025-05-12 07:34:09,855 - WARNING - Unable to validate date format: 09 Feb 75
2025-05-12 07:34:09,856 - INFO - Recognized date: 09 Feb 75 (confidence: 0.87)
2025-05-12 07:34:09,874 - INFO - Recognized hour: 15 (confidence: 0.83)
2025-05-12 07:34:09,892 - INFO - Recognized rell_number: FL740 (confidence: 0.78)
2025-05-12 07:34:09,918 - INFO - Result saved: my_results\results\nso_0223_00001_result.jpg
2025-05-12 07:34:09,920 - INFO - Processing image 3/10: https://nispdata.nso.edu/ftp/flare_patrol_h_alpha_sp/fts/223/nso_0223_00002.fts.gz
2025-05-12 07:34:09,923 - INFO - Downloading nso_0223_00002.fts.gz (attempt 1/3)
2025-05-12 07:34:15,423 - INFO - Image saved: my_results\nso_0223_00002.jpg
2025-05-12 07:34:15,427 - WARNING - Error cleaning up temp files: [WinError 32] 另一个程序正在使用此文件，进程无法访问。: 'C:\\Users\\kuina

...


0: 480x640 1 filter, 1 rell_number, 1 date, 1 hour, 1 minute, 1 seconds, 14.8ms
Speed: 3.5ms preprocess, 14.8ms inference, 2.3ms postprocess per image at shape (1, 3, 480, 640)


2025-05-12 07:34:45,196 - INFO - Recognized minute: 47 (confidence: 0.83)
2025-05-12 07:34:45,206 - WARNING - Unable to validate date format: 08 Sep 73
2025-05-12 07:34:45,207 - INFO - Recognized date: 08 Sep 73 (confidence: 0.81)
2025-05-12 07:34:45,212 - INFO - Recognized rell_number: FL746 (confidence: 0.80)
2025-05-12 07:34:45,223 - INFO - Recognized hour: 14 (confidence: 0.79)
2025-05-12 07:34:45,229 - INFO - Recognized seconds: 10 (confidence: 0.77)
2025-05-12 07:34:45,230 - WARNING - No OCR model available for filter
2025-05-12 07:34:45,301 - INFO - Result saved: my_results\results\nso_0223_00008_result.jpg
2025-05-12 07:34:45,303 - INFO - Processing image 10/10: https://nispdata.nso.edu/ftp/flare_patrol_h_alpha_sp/fts/223/nso_0223_00009.fts.gz
2025-05-12 07:34:45,304 - INFO - Downloading nso_0223_00009.fts.gz (attempt 1/3)
2025-05-12 07:34:49,136 - INFO - Image saved: my_results\nso_0223_00009.jpg
2025-05-12 07:34:49,141 - WARNING - Error cleaning up temp files: [WinError 32] 另


0: 480x640 1 rell_number, 1 date, 1 hour, 1 minute, 16.7ms
Speed: 3.3ms preprocess, 16.7ms inference, 4.5ms postprocess per image at shape (1, 3, 480, 640)


2025-05-12 07:34:49,189 - WARNING - Unable to validate date format: 09 Sep 73
2025-05-12 07:34:49,190 - INFO - Recognized date: 09 Sep 73 (confidence: 0.81)
2025-05-12 07:34:49,198 - INFO - Recognized hour: 1 (confidence: 0.65)
2025-05-12 07:34:49,210 - INFO - Recognized minute: 50 (confidence: 0.61)
2025-05-12 07:34:49,219 - INFO - Recognized rell_number: FL746 (confidence: 0.55)
2025-05-12 07:34:49,241 - INFO - Result saved: my_results\results\nso_0223_00009_result.jpg
2025-05-12 07:34:49,243 - WARNING - Unable to validate date format: 09 Feb 75
2025-05-12 07:34:49,243 - WARNING - Unable to validate date format: 08 Feb 75
2025-05-12 07:34:49,244 - WARNING - Unable to validate date format: 08 Feb 73
2025-05-12 07:34:49,246 - WARNING - Unable to validate date format: 08 Feb 75
2025-05-12 07:34:49,247 - WARNING - Unable to validate date format: 08 Feb 75
2025-05-12 07:34:49,250 - WARNING - Unable to validate date format: 05 Feb 75
2025-05-12 07:34:49,251 - WARNING - Unable to validate d

{'nso_0223_00000.fts.gz': {'recognized_data': {},
  'result_image_path': 'my_results\\results\\nso_0223_00000_result.jpg'},
 'nso_0223_00001.fts.gz': {'recognized_data': {'minute': '58',
   'date': '09 Feb 75',
   'hour': '15',
   'rell_number': 'FL740'},
  'result_image_path': 'my_results\\results\\nso_0223_00001_result.jpg'},
 'nso_0223_00002.fts.gz': {'recognized_data': {'minute': '58',
   'date': '08 Feb 75',
   'rell_number': 'FL740',
   'hour': '15',
   'filter': 'Unrecognized',
   'seconds': '20'},
  'result_image_path': 'my_results\\results\\nso_0223_00002_result.jpg'},
 'nso_0223_00003.fts.gz': {'recognized_data': {'date': '08 Feb 73',
   'filter': 'Unrecognized',
   'minute': '58',
   'rell_number': 'FL7437',
   'hour': '15',
   'seconds': '20'},
  'result_image_path': 'my_results\\results\\nso_0223_00003_result.jpg'},
 'nso_0223_00004.fts.gz': {'recognized_data': {'minute': '58',
   'rell_number': 'FL747',
   'date': '08 Feb 75',
   'hour': '15',
   'seconds': '20',
   'filt